# Lab | Agents and Tools
## A research assistant for the bank's product team

The bank from last week is planning its 2027 products, and the product team spends hours answering the same kind of questions: *How many customers are on Premium? What does the competition charge? How many complaints were about card delivery?* Some answers are in the bank's internal documents, some are on the web, and many need a quick calculation.

Your job is to build a **research assistant**: an agent that searches the internal documents, searches the web with Tavily, and does arithmetic with a calculator, and that **cites** where each fact came from. Then you **evaluate** it: does it choose the right tools, and what does each answer cost?

**The data** is `data/bank_documents.csv`: ten short internal documents of the bank (invented for this course), with the columns `id`, `title` and `text`.

**This lab practises** building and checking an agent. You will use, from today's **Agents and Tools** lesson: tools with `@tool`, `create_agent`, the message trace and its token counts, Tavily web search, a retrieval tool over Chroma, memory with a checkpointer, and step limits.

**By the end, you can:**

1. Implement an agent with three tools (a function of your own, a retrieval tool and web search) and read its trace.
2. Evaluate the agent's tool choices and cost on a test set, and explain its failures.

Each challenge ends with a **checkpoint** that tells you what you should see. Agents make their own choices, so your traces will not match the checkpoints exactly. **The whole lab costs about one US cent and fewer than 20 Tavily searches**, well within the free plan.

## Setup

You need `pip install openai python-dotenv langchain langchain-openai langchain-tavily sentence-transformers chromadb tiktoken`. Create a `.env` file in the `your-code` folder with your two keys, as in the lesson:

```
OPENAI_API_KEY=sk-...
TAVILY_API_KEY=tvly-...
```

The repository's `.gitignore` already lists `.env`: **never write a key in the notebook**, and check with `git status` before you commit that `.env` is not listed.

The cell below loads the libraries and the documents, and gives you two helpers. `run(agent, question)` runs an agent and returns a dictionary with the answer, the tools it called, the number of model calls and the tokens. `cost(input_tokens, output_tokens)` turns tokens into US dollars.

In [1]:
import os
import logging
import warnings

os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)

import pandas as pd
import tiktoken
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_tavily import TavilySearch
from langgraph.checkpoint.memory import InMemorySaver
from sentence_transformers import SentenceTransformer
import chromadb
from transformers.utils import logging as transformers_logging

transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()
warnings.filterwarnings("ignore", message=".*torch.jit.script.*")
pd.set_option("display.max_colwidth", 100)

load_dotenv()
print("OpenAI key found:", "OPENAI_API_KEY" in os.environ, "| Tavily key found:", "TAVILY_API_KEY" in os.environ)
MODEL = "openai:gpt-5.4-nano"
PRICES = (0.20, 1.25)                        # US dollars per million input / output tokens (October 2026)

def cost(input_tokens, output_tokens):
    return (input_tokens * PRICES[0] + output_tokens * PRICES[1]) / 1_000_000

def run(agent, question, config=None):
    """Run an agent on one question and summarise what it did."""
    result = agent.invoke({"messages": [{"role": "user", "content": question}]}, config or {"recursion_limit": 12})
    ai_messages = [m for m in result["messages"] if type(m).__name__ == "AIMessage"]
    tokens_in = sum(m.usage_metadata["input_tokens"] for m in ai_messages)
    tokens_out = sum(m.usage_metadata["output_tokens"] for m in ai_messages)
    return {"answer": result["messages"][-1].content,
            "tools": [call["name"] for m in ai_messages for call in m.tool_calls],
            "model_calls": len(ai_messages), "input_tokens": tokens_in, "output_tokens": tokens_out,
            "cost": cost(tokens_in, tokens_out)}

documents = pd.read_csv("../data/bank_documents.csv")
documents

OpenAI key found: True | Tavily key found: True


,id,title,text
0,premium-1,Premium plan,The Premium plan costs 9.99 euros per month. It includes card payments in foreign currencies wit...
1,customers-1,Customer base,"In September 2026 the bank had 182,000 active customers, of whom 14% were on the Premium plan. 6..."
2,complaints-1,"Complaints, third quarter of 2026","The bank received 1,240 complaints between July and September 2026: 38% about card delivery, 22%..."
3,app-1,Mobile app,In September 2026 the mobile app was rated 4.3 out of 5 in the app stores. The features customer...
4,roadmap-1,Product roadmap 2027,"Planned for 2027: an AI assistant called Nova in the app in the first quarter, budgeting tools i..."
5,fx-1,Foreign currency,Card payments in a foreign currency use the Mastercard exchange rate plus a 1.5% fee. The fee is...
6,cards-2,Card delivery,"New cards arrive within 7 working days. If a card has not arrived after 10 working days, the cus..."
7,cards-3,Blocked PIN,After three wrong PIN attempts the card is blocked. The customer can unblock it in the app under...
8,transfers-1,Transfer limits,"Customers can send up to 10,000 euros per day by SEPA transfer. Instant transfers are limited to..."
9,ai-1,AI assistant policy,"The support chatbot must start every conversation with: 'You are chatting with Nova, the bank's ..."


**Checkpoint:** both keys found, and a table of 10 documents, from `premium-1` to `ai-1`.

## Challenge 1: your first tool

1. Write a tool `calculator(expression: str) -> str` with the `@tool` decorator. Its docstring must tell the model what it does. For safety, it must **refuse** any expression that contains a character other than digits, spaces and `+ - * / ( ) .` (return an error message instead), and otherwise return the result of `eval(expression)`.
2. Test it directly with `calculator.invoke(...)` on `"182000 * 0.14"` and on `"__import__('os').getcwd()"`.
3. Create an agent with `create_agent`, the model `MODEL`, the calculator as its only tool and a short system prompt. Use `run` to ask it *What is 14% of 182,000, and what is that times 9.99?* and print the result dictionary.

In [ ]:
# 1. The calculator tool

@tool
def calculator(expression: str) -> str:
    """Evaluate an arithmetic expression such as '182000 * 0.14'.
    The expression can only contain digits, spaces, and the operators +, -, *, /, (, ), and .
    """
    # Safeguard input to ensure it only contains valid characters
    allowed_chars = set("0123456789+-*/(). ")
    if not set(expression) <= allowed_chars:
        return "Error: Expression contains illegal characters. Only numbers and basic math operators are allowed."
    
    try:
        # Evaluate safely and round to 4 decimal places
        result = eval(expression)
        return str(round(result, 4))
    except Exception as e:
        return f"Error evaluating expression: {str(e)}"


# 2. Test it directly

print("--- Direct Tool Invocations ---")
print("Valid Test:", calculator.invoke({"expression": "182000 * 0.14"}))
print("Malicious Test:", calculator.invoke({"expression": "__import__('os').getcwd()"}))

# 3. A first agent

print("\n--- Running First Agent ---")
system_prompt = (
    "You are a helpful research assistant for a banking product team. "
    "Use the calculator tool for any exact math operations or percentages. "
    "Provide clear, concise answers."
)

first_agent = create_agent(
    model=MODEL,
    tools=[calculator],
    system_prompt=system_prompt
)

# Run the agent on the evaluation question
question_1 = "What is 14% of 182,000, and what is that times 9.99?"
result_1 = run(first_agent, question_1)

# Format the printed output to check against checkpoints.
import pprint
pprint.pprint(result_1)

--- Direct Tool Invocations ---
Valid Test: 25480.0
Malicious Test: Error: Expression contains illegal characters. Only numbers and basic math operators are allowed.

--- Running First Agent ---
{'answer': '- **14% of 182,000** = **25,480**  \n'
           '- **25,480 × 9.99** = **254,545.2**',
 'cost': 0.0002258,
 'input_tokens': 504,
 'model_calls': 2,
 'output_tokens': 100,
 'tools': ['calculator', 'calculator']}


**Checkpoint:** `25480.0`, then the error message for the second expression: the tool refused to run Python code. The agent calls the calculator twice (14% of 182,000, then that times 9.99) and answers **25,480** and **254,545.2**, with three model calls, about 600 input tokens and a cost of about $0.0002.

## Challenge 2: a tool for the bank's documents

1. Load the MiniLM embedding model (`sentence-transformers/all-MiniLM-L6-v2`), embed `title. text` for every document (normalised), and store them in a Chroma collection with cosine distance and the document `id`s as ids.
2. Write a tool `search_documents(query: str) -> str` that returns the **two** closest documents, each on one line as `[id] title: text`. Its docstring must say what is in the documents, so that the agent knows when to use it.
3. Test it directly with the query *how many customers do we have*.

In [3]:
# 1. Index the documents

# Load the embedding model
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Create a local client and a collection using cosine space config
chroma_client = chromadb.Client()
policies = chroma_client.create_collection("bank_documents", configuration={"hnsw": {"space": "cosine"}})

# Prepare the data strings matching 'title: text' or 'text' pattern from the lesson
doc_texts = [f"{row['title']}: {row['text']}" for _, row in documents.iterrows()]
doc_ids = documents["id"].tolist()

# Generate normalized vector embeddings
embeddings = embedder.encode(doc_texts, normalize_embeddings=True)

# Add elements to the vector database collection
policies.add(ids=doc_ids, documents=doc_texts, embeddings=embeddings)

# 2. The retrieval tool

@tool
def search_documents(query: str) -> str:
    """Search the bank's internal documents (premium plans, customer stats, complaints, product roadmaps). 
    Use it for any question about the bank's internal data, figures, metrics, and operations."""
    # Query the closest 2 documents matching lab requirements
    result = policies.query(query_embeddings=embedder.encode([query], normalize_embeddings=True), n_results=2)
    # Format and join matching the required "[id] text" output pattern
    return "\n".join(f"[{pid}] {text}" for pid, text in zip(result["ids"][0], result["documents"][0]))

# 3. Test it

print("--- Direct Tool Invocation ---")
print(search_documents.invoke({"query": "how many customers do we have"}))

--- Direct Tool Invocation ---
[customers-1] Customer base: In September 2026 the bank had 182,000 active customers, of whom 14% were on the Premium plan. 61% of customers are aged 25 to 44.
[complaints-1] Complaints, third quarter of 2026: The bank received 1,240 complaints between July and September 2026: 38% about card delivery, 22% about blocked cards, 15% about transfers and 25% about other topics.


**Checkpoint:** two lines, `[customers-1]` first (182,000 customers, 14% on Premium) and then the next closest document, `[complaints-1]` in the reference run.

## Challenge 3: add the web, and keep the results short

1. Create a Tavily search tool with `TavilySearch(max_results=3)`. Call it once directly with the query *Revolut Premium plan price per month Spain*, and count the **tokens** of the result with `tiktoken` (use `tiktoken.get_encoding("o200k_base")` and `len(encoding.encode(str(result)))`).
2. Long tool results are expensive, and every one is sent back to the model at each later step. Write a tool `web_search(query: str) -> str` that calls the Tavily tool, keeps only the `title`, `url` and `content` of each result, and **trims** the whole text to at most 500 tokens.
3. Create the research agent with `calculator`, `search_documents` and `web_search`. Its system prompt must say when to use each tool, ask it to cite the document id or the URL of every fact, and to answer in at most four sentences.
4. Use `run` on these three questions, and print the answer, the tools and the cost of each:
   - *How much monthly revenue does the Premium plan bring?*
   - *How does our Premium plan compare with Revolut's Premium plan in price?*
   - *Name two European fintech startups that raised money for AI products in 2026, with a source for each.*

In [4]:
# 1. Tavily, and the size of one result

# Initialize the baseline TavilySearch tool with max_results=3
tavily_tool = TavilySearch(max_results=3)

# Test it directly with the query from the lab instructions
raw_search_result = tavily_tool.invoke({"query": "Revolut Premium plan price per month Spain"})

# Count the tokens of the raw string output using the o200k_base tokenizer encoding
encoding = tiktoken.get_encoding("o200k_base")
raw_token_count = len(encoding.encode(str(raw_search_result)))

print("--- Challenge 3: Raw Validation ---")
print(f"Raw Tavily output string token length: {raw_token_count} tokens")

# 2. A trimmed web search tool

@tool
def web_search(query: str) -> str:
    """Query the live web using Tavily Search for public, recent, or competitor info. 
    Use this for public pricing, external metrics, and up-to-date market facts outside bank policies."""
    # Execute the raw tool call
    raw_data = tavily_tool.invoke({"query": query})
    
    # Standard check to safeguard against empty or structural dictionary responses
    results_list = raw_data.get("results", []) if isinstance(raw_data, dict) else []
    
    # Extract only the critical string keys (title, url, content) to reduce bulk
    cleaned_items = []
    for item in results_list:
        title = item.get("title", "No Title")
        url = item.get("url", "No URL")
        content = item.get("content", "")
        cleaned_items.append(f"Title: {title}\nURL: {url}\nContent: {content}\n")
    
    cleaned_text = "\n".join(cleaned_items)
    
    # Enforce the strict 500 token ceiling on the text string chunk
    tokens = encoding.encode(cleaned_text)
    if len(tokens) > 500:
        cleaned_text = encoding.decode(tokens[:500])
        
    return cleaned_text


# 3. The research agent

print("\n--- Initializing Research Agent ---")
RESEARCH_PROMPT = (
    "You are an assistant for the bank's product team. Use search_documents for internal company files, "
    "web_search for public or competitor information, and calculator for math. "
    "Cite the explicit document id (e.g. premium-1) or the source URL you used. Answer in at most 4 sentences."
)

research_agent = create_agent(
    model=MODEL,
    tools=[calculator, search_documents, web_search],
    system_prompt=RESEARCH_PROMPT
)

# 4. Three questions

print("\n--- Evaluating Three Target Queries ---")
questions = [
    "How much monthly revenue does the Premium plan bring?",
    "How does our Premium plan compare with Revolut's Premium plan in price?",
    "Name two European fintech startups that raised money for AI products in 2026, with a source for each."
]

for idx, q in enumerate(questions, 1):
    print(f"\n[Question {idx}]: {q}")
    summary = run(research_agent, q)
    print(f"Tools called: {summary['tools']}")
    print(f"Model calls: {summary['model_calls']} | Total Cost: ${summary['cost']:.6f}")
    print(f"Answer: {summary['answer']}")
    print("-" * 50)

--- Challenge 3: Raw Validation ---
Raw Tavily output string token length: 1183 tokens

--- Initializing Research Agent ---

--- Evaluating Three Target Queries ---

[Question 1]: How much monthly revenue does the Premium plan bring?
Tools called: ['search_documents', 'calculator']
Model calls: 3 | Total Cost: $0.000431
Answer: Based on internal figures, the bank has 182,000 active customers and 14% are on the Premium plan (182,000 × 14% = 25,480 customers), and the Premium plan costs 9.99 euros per month (source: premium-1, customers-1). Monthly Premium revenue is therefore 25,480 × €9.99 ≈ **€254,545.20 per month**.
--------------------------------------------------

[Question 2]: How does our Premium plan compare with Revolut's Premium plan in price?
Tools called: ['search_documents', 'web_search']
Model calls: 2 | Total Cost: $0.000474
Answer: Our Premium plan costs **€9.99/month** ([premium-1]). Revolut’s Premium plan is **$9.99/month** in the US ([https://www.revolut.com/en-US/re

**Checkpoint:** about 900 tokens for one raw Tavily result, and 500 after trimming.

- **Revenue:** `search_documents` and `calculator`, about **254,545 euros per month**, citing `premium-1` and `customers-1`.
- **Revolut:** the documents and the web, sometimes the calculator too. Read the answer carefully: in the reference runs, the price came from another country (once in US dollars, once in UK pounds, from a comparison site) instead of Spain, and the agent said so.
- **Startups:** two or three web searches. In one reference run, the agent named two companies that are **not European**; in another, it said it could not find verifiable examples. Both are typical.

Each answer costs less than a tenth of a cent.

**Question:** why does trimming the search results matter, both for the cost and for the model? What could you lose by trimming too much?

*Your answer here.*

Trimming tool outputs is crucial for cost control because agents operate in an iterative execution loop. Since the entire conversation trace (including all previous tool responses) is sent back to the LLM as input tokens at every new step, large raw texts drastically inflate input costs exponentially over multi-step tasks. For the model, it reduces context pollution and helps keep the core query's target retrieval tokens close to the model's processing attention span. However, trimming too aggressively risks losing vital semantic context or cutting off critical data points located near the end of a long search snippet, leading the agent to incomplete or factually broken final answers.

## Challenge 4: evaluate the tool choices

The product team wrote eight questions and, for each, the set of tools a good answer needs.

1. Run `research_agent` on every question in `TEST_SET` with `run`, and build a DataFrame with the question, the expected tools, the tools called (as a set), whether they are **exactly** the expected set (`right_tools`), the number of model calls, the tokens and the cost.
2. Print the share of questions with the right tools, the total cost, and the average cost of the questions that used `web_search` against those that did not.
3. Read the answers of the questions with the wrong tools.

In [5]:
TEST_SET = [
    ("What does the Premium plan include?", {"search_documents"}),
    ("How many customers are on the Premium plan?", {"search_documents", "calculator"}),
    ("How many complaints were about card delivery last quarter?", {"search_documents", "calculator"}),
    ("What is 17.5% of 2,400?", {"calculator"}),
    ("What is the monthly price of Revolut's Premium plan in Spain?", {"web_search"}),
    ("What did the European Central Bank decide about interest rates at its latest meeting?", {"web_search"}),
    ("When will our app get an AI assistant, and what must it tell users under the EU AI Act?", {"search_documents", "web_search"}),
    ("Do Premium customers pay a fee on card payments abroad, and how much would a standard customer pay on 300 euros?", {"search_documents", "calculator"}),
]

# 1. Run the test set

rows = []

for question, expected_tools in TEST_SET:
    # Run the agent on the evaluation question
    summary = run(research_agent, question)
    
    # Extract data tokens and tool selection list
    tools_called_set = set(summary["tools"])
    is_correct = tools_called_set == expected_tools
    
    rows.append({
        "question": question,
        "expected_tools": expected_tools,
        "tools_called": tools_called_set,
        "right_tools": is_correct,
        "model_calls": summary["model_calls"],
        "input_tokens": summary["input_tokens"],
        "output_tokens": summary["output_tokens"],
        "cost": summary["cost"]
    })

# Convert to DataFrame for analytical scanning
df_eval = pd.DataFrame(rows)

# 2. Right tools and cost

share_correct = df_eval["right_tools"].mean()
total_cost = df_eval["cost"].sum()

# Compute conditional means based on whether web_search was invoked
used_web = df_eval[df_eval["tools_called"].apply(lambda s: "web_search" in s)]
no_web = df_eval[df_eval["tools_called"].apply(lambda s: "web_search" not in s)]

avg_cost_web = used_web["cost"].mean() if not used_web.empty else 0.0
avg_cost_no_web = no_web["cost"].mean() if not no_web.empty else 0.0

print("--- Evaluation Performance Metrics ---")
print(f"Share of questions with exactly the right tools: {share_correct * 100:.1f}%")
print(f"Total test suite cost: ${total_cost:.6f}")
print(f"Average cost with web search:    ${avg_cost_web:.6f}")
print(f"Average cost without web search: ${avg_cost_no_web:.6f}\n")




# 3. The wrong ones

print("--- Detailed Evaluation Matrix ---")
display(df_eval[["question", "right_tools", "model_calls", "cost"]])

--- Evaluation Performance Metrics ---
Share of questions with exactly the right tools: 87.5%
Total test suite cost: $0.002866
Average cost with web search:    $0.000456
Average cost without web search: $0.000299

--- Detailed Evaluation Matrix ---


,question,right_tools,model_calls,cost
0,What does the Premium plan include?,True,2,0.000247
1,How many customers are on the Premium plan?,True,3,0.000369
2,How many complaints were about card delivery last quarter?,False,2,0.000261
3,"What is 17.5% of 2,400?",True,2,0.000189
4,What is the monthly price of Revolut's Premium plan in Spain?,True,2,0.000318
5,What did the European Central Bank decide about interest rates at its latest meeting?,True,2,0.000385
6,"When will our app get an AI assistant, and what must it tell users under the EU AI Act?",True,2,0.000666
7,"Do Premium customers pay a fee on card payments abroad, and how much would a standard customer p...",True,3,0.000431


**Checkpoint:** 7 of the 8 questions (88%) with exactly the expected tools, for a total of about a third of a cent; in the reference run the web questions cost a little more on average (about $0.0004 against $0.0003, thanks to the trimming). The miss was the card-delivery question: the agent found `complaints-1` and computed 38% of 1,240 = 471 **by itself**, without the calculator.

**Question:** look at the questions where the tools were not exactly the expected ones. Was the answer wrong, or was the test too strict? What would you change: the tool descriptions, the system prompt, or the test?

*Your answer here.*

The answer itself was completely correct; the agent successfully retrieved the document complaints-1 and accurately deduced that 38% of 1,240 is 471. However, the evaluation test was too strict because it assumed the agent would explicitly call the calculator tool to do this basic math.
Since modern LLMs possess internal parametric capabilities for clean percentages, it bypassed the tool to optimize the steps. To fix this, I would modify the system prompt to state: "You must always explicitly call the calculator tool for any mathematical operation or percentage calculation, no matter how simple." Alternatively, the evaluation test could be updated to accept either tool path as valid.

## Challenge 5: memory and limits

1. Create `chat_agent`: the same tools and system prompt, plus `checkpointer=InMemorySaver()`.
2. Hold this conversation in one thread (`{"configurable": {"thread_id": "product-team"}, "recursion_limit": 12}`), printing each answer:
   - *How many complaints did we get last quarter?*
   - *What share of them were about blocked cards, and how many is that?*
   - *Remind me: what was the total you started from?*
3. Ask the research agent *Compare every one of our documents with what our three main competitors offer, one by one.* with `{"recursion_limit": 4}`, and catch the `GraphRecursionError` (import it from `langgraph.errors`). What should a real application show the user when this happens?

In [8]:
from langgraph.errors import GraphRecursionError
# 1. An agent with memory

memory_saver = InMemorySaver()

chat_agent = create_agent(
    model=MODEL,
    tools=[calculator, search_documents, web_search],
    system_prompt=RESEARCH_PROMPT,
    checkpointer=memory_saver
)

# 2. A conversation in one thread

print("--- Challenge 5: Conversation Thread Validation ---")
thread_config = {"configurable": {"thread_id": "product-team"}, "recursion_limit": 12}

chat_questions = [
    "How many complaints did we get last quarter?",
    "What share of them were about blocked cards, and how many is that?",
    "Remind me: what was the total you started from?"
]

for q in chat_questions:
    print(f"\nUser: {q}")
    res = chat_agent.invoke({"messages": [{"role": "user", "content": q}]}, thread_config)
    print(f"Agent: {res['messages'][-1].content}")


# 3. A step limit

print("\n--- Challenge 5: Step Limit Safeguard ---")
broad_question = "Compare every one of our documents with what our three main competitors offer, one by one."

try:
    # Pass BOTH the required thread config and the low step limit
    limit_config = {
        "configurable": {"thread_id": "step-limit-test-thread"}, 
        "recursion_limit": 4
    }
    chat_agent.invoke({"messages": [{"role": "user", "content": broad_question}]}, limit_config)

except (GraphRecursionError, ValueError) as e:
    # Now this will cleanly intercept the true recursion-limit step wall error!
    if "recursion" in str(e).lower() or isinstance(e, GraphRecursionError):
        print("Caught expected error: Stopped by the step limit.")
    else:
        # If it's a different ValueError, raise it so we can debug it
        raise e


--- Challenge 5: Conversation Thread Validation ---

User: How many complaints did we get last quarter?
Agent: We received **1,240 complaints** last quarter (July–September 2026). (Source: **complaints-1**)

User: What share of them were about blocked cards, and how many is that?
Agent: **22%** of complaints were about **blocked cards**, which is about **273 complaints** (22% of 1,240). (Sources: **complaints-1**)

User: Remind me: what was the total you started from?
Agent: You started from a total of **1,240 complaints**. (Source: **complaints-1**)

--- Challenge 5: Step Limit Safeguard ---
Caught expected error: Stopped by the step limit.


**Checkpoint:** 1,240 complaints; then 22%, about 273 (272.8); then 1,240 again, which proves the memory works. The broad question prints `Stopped by the step limit`. A real application should say that the question was too broad and suggest a narrower one (one product, one competitor), not show a half-finished answer, and it should log the case.

## What did it cost?

Add up the cost of the evaluation in Challenge 4 and compare it with the price of the team's time.

**Question:** the test set cost a fraction of a cent per question. If the product team asks 200 questions a day, about how much would the assistant cost per month? Which part of the cost would you watch, and what is the cost you cannot see in the token count?

*Your answer here.*
• Monthly API Cost Estimation: Based on our test suite evaluation metrics, the average query overhead runs around $0.00036. At a velocity of 200 user queries per day, the immediate cost totals roughly $0.072 daily, compounding to approximately $2.16 per month for the entire core product team.

• Cost Drivers to Monitor Closely: The most critical metrics to watch are the Input Tokens within long-running conversation threads. Because checkpointers pass the cumulative message list back to the model on every single turn, a thread with an extensive chat history becomes progressively more expensive with every follow-up question.

• Hidden Costs Outside the Token Count: The hidden production factors not visible in standard token metrics include Tavily Search API pricing tiers (which will quickly exceed the 1,000 free monthly queries limit under standard team usage), the database computing infrastructure needed to host cloud-based state savers, and internal developer resource maintenance/observability costs.

## Extra: give the assistant a voice

From the **Multimodal AI** lesson:

1. Generate a spoken question with `gpt-4o-mini-tts`, for example *How many of our customers are on the Premium plan?*, and save it as `question.mp3`.
2. Transcribe it with `gpt-4o-mini-transcribe`, send the transcript to `research_agent`, and turn the answer into speech.
3. Listen to the answer. Is a four-sentence answer with citations a good spoken answer? Change the system prompt for a voice assistant and try again.

In [ ]:
# Your code here

**Conclusions:** *Your answer here.*